# Notebook 01: Preparación de datos
**Proyecto:** Predicción de Insatisfacción del Cliente — Olist (Grupo C)

**Objetivo:** construir una tabla con **una fila por pedido**, que contenga la variable objetivo
(`insatisfecho`) y las variables explicativas, usando solo información disponible en el
**momento de predicción**.

**Pasos:**
1. Carga de los 7 datasets y traducción al español.
2. Variable objetivo y reseñas duplicadas.
3. Población de estudio y momento de predicción.
4. Variables de tiempo, de ítems, de pago, de ubicación y de historial del vendedor.
5. Guardado de la base consolidada.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Las funciones propias están en la carpeta src (una carpeta arriba del notebook)
sys.path.append(os.path.abspath('../src'))
from preparacion import cargar_crudos, traducir, dif_dias, tasa_insatisfaccion, historial_vendedor

pd.set_option('display.max_columns', 60)

RUTA_DATOS = '../data'
RUTA_PROC = '../data/procesado'
os.makedirs(RUTA_PROC, exist_ok=True)

## 1. Carga y traducción
Se cargan los 7 datasets originales. La función `cargar_crudos` (en `src/preparacion.py`) convierte
las fechas y lee los códigos postales como texto para no perder los ceros iniciales.

In [2]:
dfs = cargar_crudos(RUTA_DATOS)

for nombre, df in dfs.items():
    print(f'{nombre:10}: {df.shape[0]:>7} filas, {df.shape[1]:>2} columnas')

orders    :   99441 filas,  8 columnas
items     :  112650 filas,  7 columnas
payments  :  103886 filas,  5 columnas
reviews   :   99224 filas,  7 columnas
customers :   99441 filas,  5 columnas
products  :   32951 filas,  9 columnas
sellers   :    3095 filas,  4 columnas


### ¿Qué se traduce?
| Variable | Idioma original | Decisión |
|---|---|---|
| Nombres de columnas | Inglés | Se mantienen |
| `product_category_name` | Portugués | Se traduce al español (73 categorías, diccionario propio) |
| `order_status`, `payment_type` | Inglés | Se traducen al español (uniformidad en el informe) |
| `customer_state`, `seller_state` | Sigla | Se agrega la región de Brasil |
| `customer_city`, `seller_city` | Portugués | No se traducen (nombres propios) |
| `review_comment_message` | Portugués | No se traduce: se escribe junto con la puntuación (fuga de información) |

Los diccionarios están en `src/diccionarios.py`. El archivo de traducción del dataset
(al inglés) se descartó porque le faltaban 2 categorías y tenía errores de tipeo.

In [3]:
dfs = traducir(dfs)

# Controles: ningún valor debe quedar sin traducir
p = dfs['products']
print('Categorías sin traducir:', ((p['categoria_producto'] == 'sin_categoria')
                                   & p['product_category_name'].notna()).sum())
print('Productos sin categoría (nulos de origen):', p['product_category_name'].isna().sum())
print('Estados de pedido sin traducir:', dfs['orders']['estado_pedido'].isna().sum())
print('Tipos de pago sin traducir:', dfs['payments']['tipo_pago'].isna().sum())
print('Clientes sin región:', dfs['customers']['customer_region'].isna().sum())
print('Vendedores sin región:', dfs['sellers']['seller_region'].isna().sum())

# Cada tabla en su propia variable para que el código siguiente sea más legible
orders, items, payments = dfs['orders'], dfs['items'], dfs['payments']
reviews, customers = dfs['reviews'], dfs['customers']
products, sellers = dfs['products'], dfs['sellers']

Categorías sin traducir: 0
Productos sin categoría (nulos de origen): 610
Estados de pedido sin traducir: 0
Tipos de pago sin traducir: 0
Clientes sin región: 0
Vendedores sin región: 0


## 2. Variable objetivo y reseñas duplicadas
- **insatisfecho = 1** si `review_score` está entre 1 y 3; **0** si es 4 o 5 (según la propuesta).
- Los indicadores de comentario se crean **solo para el EDA**: se escriben junto con la puntuación.
- Algunos pedidos tienen varias reseñas: se conserva la **más reciente** (última opinión del cliente).

In [4]:
reviews = dfs['reviews'].copy()

reviews['insatisfecho'] = (reviews['review_score'] <= 3).astype(int)
reviews['tiene_comentario'] = reviews['review_comment_message'].notna().astype(int)
reviews['largo_comentario'] = reviews['review_comment_message'].fillna('').str.len()

# El texto ya no se usa: se elimina para aliviar la memoria
reviews = reviews.drop(columns=['review_comment_title', 'review_comment_message'])

# Pedidos con más de una reseña
cant = reviews['order_id'].value_counts()
pedidos_varias = cant[cant > 1].index
dup = reviews[reviews['order_id'].isin(pedidos_varias)]

# Contradictorias = el mismo pedido tiene una reseña satisfecha y otra insatisfecha
contradictorias = dup.groupby('order_id')['insatisfecho'].nunique() > 1

print(f'Reseñas totales: {len(reviews):,} para {reviews["order_id"].nunique():,} pedidos')
print(f'Pedidos con más de una reseña: {len(pedidos_varias):,}')
print(f'  - Reseñas que coinciden (todas satisfechas o todas insatisfechas): {(~contradictorias).sum():,}')
print(f'  - Reseñas contradictorias (una satisfecha y otra insatisfecha):   {contradictorias.sum():,}')

# Ejemplo de un pedido con reseñas contradictorias
ejemplo = contradictorias[contradictorias].index[0]
print('\nEjemplo de pedido con reseñas contradictorias:')
display(reviews.loc[reviews['order_id'] == ejemplo,
                    ['review_score', 'insatisfecho', 'review_answer_timestamp']]
               .sort_values('review_answer_timestamp'))

# Regla: se conserva la reseña más reciente de cada pedido
reviews = (reviews.sort_values('review_answer_timestamp')
                  .drop_duplicates('order_id', keep='last'))
print(f'Reseñas finales: {len(reviews):,} (una por pedido: {reviews["order_id"].is_unique})')

Reseñas totales: 99,224 para 98,673 pedidos
Pedidos con más de una reseña: 547
  - Reseñas que coinciden (todas satisfechas o todas insatisfechas): 429
  - Reseñas contradictorias (una satisfecha y otra insatisfecha):   118

Ejemplo de pedido con reseñas contradictorias:


,review_score,insatisfecho,review_answer_timestamp
69438,3,1,2018-03-06 19:50:32
8273,3,1,2018-03-21 02:28:23
51527,4,0,2018-03-30 00:29:09


Reseñas finales: 98,673 (una por pedido: True)


## 3. Población de estudio
Se analiza la insatisfacción según el estado del pedido.

In [5]:
tmp = orders.merge(reviews[['order_id', 'insatisfecho']], on='order_id')

tabla = (tmp.groupby('estado_pedido')['insatisfecho']
            .agg(['size', 'mean'])
            .rename(columns={'size': 'pedidos', 'mean': '% insatisfechos'})
            .sort_values('pedidos', ascending=False))
tabla['% del total'] = (tabla['pedidos'] / tabla['pedidos'].sum() * 100).round(1)
tabla['% insatisfechos'] = (tabla['% insatisfechos'] * 100).round(1)
tabla = tabla[['pedidos', '% del total', '% insatisfechos']]
tabla.index.name = 'estado del pedido'
display(tabla)

,pedidos,% del total,% insatisfechos
estado del pedido,,,
entregado,95832,97.1,21.1
enviado,1032,1.0,80.1
cancelado,605,0.6,84.8
no_disponible,595,0.6,90.8
facturado,309,0.3,88.0
en_proceso,295,0.3,95.9
creado,3,0.0,66.7
aprobado,2,0.0,50.0


**Decisión:** se trabaja con pedidos **entregados**, con fecha de entrega y con reseña.
- Los no entregados tienen 80–96% de insatisfacción: el resultado es obvio y no requiere un modelo.
- Las variables de entrega solo existen si el pedido llegó.
- Sin reseña no hay variable objetivo (unión interna).

In [6]:
base = orders[(orders['estado_pedido'] == 'entregado') &
              orders['order_delivered_customer_date'].notna()]

base = base.merge(reviews[['order_id', 'review_score', 'insatisfecho', 'tiene_comentario',
                           'largo_comentario', 'review_creation_date', 'review_answer_timestamp']],
                  on='order_id', how='inner')
print('Pedidos entregados con reseña:', len(base))
print('¿Un pedido por fila?', base['order_id'].is_unique)

Pedidos entregados con reseña: 95824
¿Un pedido por fila? True


### Momento de predicción
El modelo debe predecir la insatisfacción **antes de que el cliente responda la encuesta**.
Olist envía la encuesta cuando ocurre lo primero entre:
- la **entrega** del pedido, o
- la **fecha estimada de entrega** (si el pedido todavía no llegó).

Por eso se define: **momento de predicción = mínimo entre la fecha de entrega y la fecha estimada**.
Todas las variables predictoras deben conocerse en ese momento. Por ejemplo, si el pedido no
llegó para la fecha estimada, se sabe que está **atrasado**, pero todavía no **cuántos días** se va a atrasar.

In [7]:
# Momento de predicción: lo primero entre la entrega real y la fecha estimada
base['momento_prediccion'] = base[['order_delivered_customer_date',
                                   'order_estimated_delivery_date']].min(axis=1)

# Verificación: ¿cuántos días después del momento de predicción se crea la encuesta?
dias = (base['review_creation_date'] - base['momento_prediccion'].dt.normalize()).dt.days

etiquetas = {-1: 'antes del momento', 0: 'el mismo día', 1: '1 día después',
             2: '2 días después', 3: '3 días después', 4: '4 o más días después'}
resumen = (dias.clip(-1, 4)
               .value_counts().sort_index()
               .rename(index=etiquetas).to_frame('cantidad de pedidos'))
resumen['% de pedidos'] = (resumen['cantidad de pedidos'] / resumen['cantidad de pedidos'].sum() * 100).round(1)
resumen.index.name = 'Creación de la encuesta respecto del momento de predicción'
display(resumen)

# Se excluyen las reseñas respondidas antes del momento de predicción (casos atípicos)
resena_previa = base['review_answer_timestamp'] < base['momento_prediccion']
print(f'Reseñas respondidas antes del momento de predicción (se excluyen): {resena_previa.sum():,}')

base = base[~resena_previa].copy()
print(f'Pedidos en la base: {len(base):,} | tasa de insatisfacción: {base["insatisfecho"].mean():.1%}')

,cantidad de pedidos,% de pedidos
Creación de la encuesta respecto del momento de predicción,,
antes del momento,279,0.3
el mismo día,2588,2.7
1 día después,84616,88.3
2 días después,5419,5.7
3 días después,1081,1.1
4 o más días después,1841,1.9


Reseñas respondidas antes del momento de predicción (se excluyen): 221
Pedidos en la base: 95,603 | tasa de insatisfacción: 21.0%


**Conclusión:** en el 88,3% de los pedidos la encuesta se crea al día siguiente del momento de
predicción, y en el 99,7% ese mismo día o después. Esto confirma que, en el momento de predicción,
el cliente todavía no opinó. Se excluyen 221 pedidos cuya reseña se respondió antes (casos atípicos).
Base final: **95.603 pedidos**, con **21,0%** de insatisfechos.

## 4. Variables de tiempo
Todas las variables predictoras se calculan **hasta el momento de predicción**.

| Variable | Cálculo | Interpretación |
|---|---|---|
| `entrega_tardia` | 1 si el pedido llegó **después del día** estimado | ¿Está atrasado? (se sabe con certeza en el momento de predicción) |
| `dias_entrega_observados` | momento de predicción − compra | Días que tardó; si está atrasado, se cuenta hasta la fecha estimada |
| `dias_anticipacion` | fecha estimada − momento de predicción | Cuántos días **antes** de lo prometido llegó (0 si está atrasado) |
| `dias_entrega_estimada` | fecha estimada − compra | Plazo prometido al cliente |
| `horas_aprobacion` | aprobación del pago − compra | Demora en aprobar el pago |
| `dias_hasta_transportista` | despacho al transportista − compra | Demora del vendedor en despachar (hasta el momento de predicción) |
| `mes_compra`, `dia_semana_compra` | de la fecha de compra | Estacionalidad |

**Solo para el EDA** (no predictoras): `dias_entrega_real` y `dias_retraso`. En los pedidos
atrasados su valor se conoce recién cuando llega el pedido, después de la encuesta.

In [8]:
# Despacho al transportista "visible" en el momento de predicción
# (si ocurrió después, en ese momento todavía no se había despachado)
base['despacho_observado'] = base[['order_delivered_carrier_date', 'momento_prediccion']].min(axis=1)

# --- Variables predictoras: conocidas en el momento de predicción
# Se compara por DÍA: la fecha estimada viene sin hora (00:00), así que un pedido
# que llega ese mismo día a la tarde NO cuenta como atrasado
base['entrega_tardia'] = (base['order_delivered_customer_date'].dt.normalize()
                          > base['order_estimated_delivery_date']).astype(int)

base['dias_entrega_observados']  = dif_dias(base, 'momento_prediccion', 'order_purchase_timestamp')
base['dias_anticipacion']        = dif_dias(base, 'order_estimated_delivery_date', 'momento_prediccion')
base['dias_entrega_estimada']    = dif_dias(base, 'order_estimated_delivery_date', 'order_purchase_timestamp')
base['horas_aprobacion']         = dif_dias(base, 'order_approved_at', 'order_purchase_timestamp') * 24
base['dias_hasta_transportista'] = dif_dias(base, 'despacho_observado', 'order_purchase_timestamp')
base['mes_compra']               = base['order_purchase_timestamp'].dt.month
base['dia_semana_compra']        = base['order_purchase_timestamp'].dt.day_of_week   # 0 = lunes

# --- Variables reales completas: SOLO para el EDA
base['dias_entrega_real'] = dif_dias(base, 'order_delivered_customer_date', 'order_purchase_timestamp')
base['dias_retraso']      = dif_dias(base, 'order_delivered_customer_date', 'order_estimated_delivery_date')

vars_tiempo = ['dias_entrega_observados', 'dias_anticipacion', 'dias_entrega_estimada',
               'horas_aprobacion', 'dias_hasta_transportista']
display(base[vars_tiempo].describe().round(2).T)

,count,mean,std,min,25%,50%,75%,max
dias_entrega_observados,95603.0,11.74,7.00,0.53,6.72,10.16,15.40,70.21
dias_anticipacion,95603.0,11.98,7.90,0.00,6.41,11.98,16.25,146.02
dias_entrega_estimada,95603.0,23.72,8.75,2.01,18.32,23.22,28.40,155.14
horas_aprobacion,95589.0,10.27,20.53,0.00,0.22,0.34,14.51,741.44
dias_hasta_transportista,95603.0,3.19,3.38,-171.21,1.13,2.20,4.06,55.10


In [9]:
display(tasa_insatisfaccion(base, 'entrega_tardia', {0: 'a tiempo', 1: 'atrasado'}))

,cantidad de pedidos,% de pedidos,% insatisfechos
entrega_tardia,,,
a tiempo,89263,93.4,17.3
atrasado,6340,6.6,73.5


## Fechas inconsistentes
Se marcan los pedidos cuyas fechas no siguen el orden lógico
(compra → aprobación → despacho → entrega). No se eliminan: se tratan en el preprocesamiento.

In [10]:
base['fecha_inconsistente'] = (
    (base['order_approved_at'] < base['order_purchase_timestamp'])            # aprobado antes de comprar
    | (base['order_delivered_carrier_date'] < base['order_approved_at'])      # despachado antes de aprobar
    | (base['order_delivered_customer_date'] < base['order_delivered_carrier_date'])  # entregado antes de despachar
).astype(int)

print(f'Pedidos con fechas inconsistentes: {base["fecha_inconsistente"].sum():,} '
      f'({base["fecha_inconsistente"].mean():.1%} de la base)')

Pedidos con fechas inconsistentes: 1,363 (1.4% de la base)


## 5. Variables de ítems y productos (agregadas por pedido)
Un pedido puede tener varios ítems: se resumen por pedido (sumas, conteos y promedios).
La **categoría principal** y el **vendedor principal** son los del ítem más caro del pedido.
Se agrega la ubicación del vendedor (tabla `sellers`) para compararla con la del cliente.

In [11]:
cols_prod = ['product_id', 'categoria_producto', 'product_weight_g', 'product_length_cm',
             'product_height_cm', 'product_width_cm', 'product_photos_qty', 'product_description_lenght']

# Cada ítem con los datos de su producto y la ubicación de su vendedor
items_prod = (items
              .merge(products[cols_prod], on='product_id', how='left')
              .merge(sellers[['seller_id', 'seller_state', 'seller_region']], on='seller_id', how='left'))
items_prod['volumen_cm3'] = (items_prod['product_length_cm'] * items_prod['product_height_cm']
                             * items_prod['product_width_cm'])

# Resumen por pedido
items_agg = items_prod.groupby('order_id').agg(
    cant_items=('order_item_id', 'count'),
    cant_productos=('product_id', 'nunique'),
    cant_vendedores=('seller_id', 'nunique'),
    precio_total=('price', 'sum'),
    flete_total=('freight_value', 'sum'),
    peso_total_g=('product_weight_g', 'sum'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    largo_descripcion_promedio=('product_description_lenght', 'mean'),
    fecha_limite_despacho=('shipping_limit_date', 'max'),
).reset_index()

# Proporción del flete sobre el precio (flete caro en relación al producto)
items_agg['ratio_flete_precio'] = items_agg['flete_total'] / items_agg['precio_total']

# Categoría, vendedor y ubicación del vendedor del ítem más caro de cada pedido
principal = (items_prod.sort_values(['order_id', 'price'], ascending=[True, False])
                       .drop_duplicates('order_id')
                       [['order_id', 'categoria_producto', 'seller_id', 'seller_state', 'seller_region']]
                       .rename(columns={'categoria_producto': 'categoria_principal',
                                        'seller_id': 'vendedor_principal'}))
items_agg = items_agg.merge(principal, on='order_id')

print(f'Pedidos con ítems: {len(items_agg):,} | ¿un pedido por fila? {items_agg["order_id"].is_unique}')

Pedidos con ítems: 98,666 | ¿un pedido por fila? True


## 6. Variables de pago (agregadas por pedido)
Un pedido puede pagarse con más de un medio (ej.: tarjeta + voucher).
El **tipo de pago principal** es el de mayor monto.

In [12]:
pagos_agg = payments.groupby('order_id').agg(
    cant_medios_pago=('payment_sequential', 'count'),
    cuotas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
).reset_index()

tipo_principal = (payments.sort_values(['order_id', 'payment_value'], ascending=[True, False])
                          .drop_duplicates('order_id')[['order_id', 'tipo_pago']])
pagos_agg = pagos_agg.merge(tipo_principal, on='order_id')

print(f'Pedidos con pagos: {len(pagos_agg):,} | ¿un pedido por fila? {pagos_agg["order_id"].is_unique}')

Pedidos con pagos: 99,440 | ¿un pedido por fila? True


## 7. Unión de las tablas y ubicación cliente-vendedor
Se agregan a la base: ítems, pagos y cliente. Además:
- `despacho_tardio`: 1 si el vendedor despachó después de la fecha límite que tenía.
- `mismo_estado` / `misma_region`: 1 si cliente y vendedor principal están en el mismo estado / región.
  Aproximan la **distancia comprador-vendedor** de la propuesta (no se usa la tabla de geolocalización).

In [13]:
df = (base
      .merge(items_agg, on='order_id', how='left')
      .merge(pagos_agg, on='order_id', how='left')
      .merge(customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_region']],
             on='customer_id', how='left'))

# Demora del vendedor respecto de su fecha límite de despacho (usando el despacho observado)
df['dias_despacho_vs_limite'] = dif_dias(df, 'despacho_observado', 'fecha_limite_despacho')
df['despacho_tardio'] = (df['dias_despacho_vs_limite'] > 0).astype(int)

# Ubicación cliente-vendedor
df['mismo_estado'] = (df['customer_state'] == df['seller_state']).astype(int)
df['misma_region'] = (df['customer_region'] == df['seller_region']).astype(int)

print(f'Filas: {len(df):,} | ¿un pedido por fila? {df["order_id"].is_unique}')

Filas: 95,603 | ¿un pedido por fila? True


In [14]:
si_no = {0: 'no', 1: 'sí'}

print('Despacho tardío del vendedor:')
display(tasa_insatisfaccion(df, 'despacho_tardio', si_no))

print('Cliente y vendedor en el mismo estado:')
display(tasa_insatisfaccion(df, 'mismo_estado', si_no))

print('Cliente y vendedor en la misma región:')
display(tasa_insatisfaccion(df, 'misma_region', si_no))

Despacho tardío del vendedor:


,cantidad de pedidos,% de pedidos,% insatisfechos
despacho_tardio,,,
no,87073,91.1,19.8
sí,8530,8.9,34.0


Cliente y vendedor en el mismo estado:


,cantidad de pedidos,% de pedidos,% insatisfechos
mismo_estado,,,
no,61174,64.0,22.7
sí,34429,36.0,18.1


Cliente y vendedor en la misma región:


,cantidad de pedidos,% de pedidos,% insatisfechos
misma_region,,,
no,36122,37.8,22.4
sí,59481,62.2,20.2


## 8. Historial del vendedor (sin fuga de información)
Para cada pedido se calcula el historial de su vendedor principal usando **solo información
disponible al momento de la compra** (función `historial_vendedor` en `src/preparacion.py`):

| Variable | Qué mide |
|---|---|
| `vendedor_pedidos_previos` | Pedidos que el vendedor recibió antes (experiencia / volumen) |
| `vendedor_tasa_retraso_previa` | % de entregas tardías entre sus pedidos ya entregados |
| `vendedor_tasa_insatisf_previa` | % de clientes insatisfechos entre sus reseñas ya respondidas |

Si el vendedor todavía no tiene historial, las tasas quedan nulas (se tratan en el preprocesamiento).

In [15]:
# Se aplica vendedor por vendedor y se vuelve a unir
df = pd.concat([historial_vendedor(g) for _, g in df.groupby('vendedor_principal')])
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)

vars_vendedor = ['vendedor_pedidos_previos', 'vendedor_tasa_retraso_previa', 'vendedor_tasa_insatisf_previa']
display(df[vars_vendedor].describe().round(3).T)

,count,mean,std,min,25%,50%,75%,max
vendedor_pedidos_previos,95603.0,180.909,292.253,0.0,15.00,58.000,197.000,1786.0
vendedor_tasa_retraso_previa,90375.0,0.043,0.065,0.0,0.00,0.030,0.059,1.0
vendedor_tasa_insatisf_previa,89739.0,0.199,0.131,0.0,0.13,0.193,0.252,1.0


In [16]:
tramos = pd.cut(df['vendedor_tasa_insatisf_previa'], [-0.01, 0.1, 0.2, 0.3, 0.5, 1],
                labels=['0-10%', '10-20%', '20-30%', '30-50%', 'más de 50%'])
tabla = tasa_insatisfaccion(df.assign(tramo=tramos), 'tramo')
tabla.index.name = 'insatisfacción previa del vendedor'
display(tabla)

,cantidad de pedidos,% de pedidos,% insatisfechos
insatisfacción previa del vendedor,,,
0-10%,16045,17.9,17.6
10-20%,33186,37.0,19.1
20-30%,28171,31.4,22.8
30-50%,10823,12.1,27.2
más de 50%,1514,1.7,30.5


## 9. Guardado de la base consolidada
Las columnas se agrupan según su uso en los notebooks siguientes:

| Grupo | Columnas | Uso |
|---|---|---|
| Identificadores y fechas | `order_id`, `customer_id`, `vendedor_principal`, fechas originales, `momento_prediccion` | Referencia: no son predictoras |
| Variable objetivo | `insatisfecho` | Lo que se predice |
| **Solo EDA (fuga)** | `review_score`, `tiene_comentario`, `largo_comentario`, fechas de la reseña, `dias_entrega_real`, `dias_retraso` | Se conocen **después** del momento de predicción: **nunca** como predictoras |
| Predictoras | todas las demás | Candidatas para los modelos |

In [17]:
print(f'Filas: {len(df):,} | Columnas: {df.shape[1]} | ¿un pedido por fila? {df["order_id"].is_unique}')
print(f'Tasa de insatisfacción: {df["insatisfecho"].mean():.1%}')

print('\nVariables con nulos:')
nulos = df.isna().sum()
display(nulos[nulos > 0].sort_values(ascending=False).to_frame('cantidad de nulos'))

df.to_csv(os.path.join(RUTA_PROC, 'base_consolidada.csv'), index=False)
print('Guardado en', os.path.join(RUTA_PROC, 'base_consolidada.csv'))

Filas: 95,603 | Columnas: 57 | ¿un pedido por fila? True
Tasa de insatisfacción: 21.0%

Variables con nulos:


,cantidad de nulos
vendedor_tasa_insatisf_previa,5864
vendedor_tasa_retraso_previa,5228
fotos_promedio,1321
largo_descripcion_promedio,1321
horas_aprobacion,14
order_approved_at,14
cuotas,1
cant_medios_pago,1
tipo_pago,1
valor_pagado,1


Guardado en ../data/procesado\base_consolidada.csv


## Resumen
- **Base final:** 95.603 pedidos entregados con reseña, una fila por pedido, **21,0% insatisfechos**.
- **Momento de predicción:** el primero entre la entrega y la fecha estimada (la encuesta se crea
  al día siguiente en el 88% de los casos). Ninguna variable predictora usa información posterior.
- **Hallazgos para el EDA:**
  - Pedido atrasado: 73,5% de insatisfacción vs 17,3% a tiempo.
  - Despacho tardío del vendedor: 34,0% vs 19,8%.
  - Vendedor del mismo estado: 18,1% vs 22,7%.
  - Mal historial del vendedor (>50%): 30,5% vs 17,6% (historial ≤10%).
- **Pendiente para el preprocesamiento:** nulos (historial de vendedores nuevos, fotos y descripción),
  1.363 pedidos con fechas inconsistentes, outliers en tiempos y montos.

**Próximo paso → Notebook 02: EDA** (inciso 1 del enunciado).